In [41]:
!pip install numpy pandas scikit-learn demoji textblob
!python -m textblob.download_corpora

[nltk_data] Downloading package brown to /home/saatvik/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/saatvik/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/saatvik/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/saatvik/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to
[nltk_data]     /home/saatvik/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     /home/saatvik/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
Finished.


In [42]:
import numpy as np
import pandas as pd
import sklearn as skl
import demoji as dem
import textblob as tb
dem.download_codes()


/tmp/ipykernel_14900/3331246871.py:6: FutureWarning: The demoji.download_codes attribute is deprecated and will be removed from demoji in a future version. It is an unused attribute as emoji codes are now distributed directly with the demoji package.
  dem.download_codes()


# Reading Data & Data Preprocessing

In [43]:
df = pd.read_csv("semeval-tweets/twitter-training-data.txt", sep="\t", header=None, names=["id", "sentiment", "tweet"])
print(f"NaN tweets: {df['tweet'].isna().sum()}")
df = df.dropna(subset=["tweet"]).reset_index(drop=True)

for i in range(len(df["tweet"])):
  tweet = df.loc[i, "tweet"]
  tweetNoEmoji = dem.replace_with_desc(tweet)
  df.loc[i, "tweet"] = tweetNoEmoji
df

NaN tweets: 1


,id,sentiment,tweet
0,335104872099066692,positive,Felt privileged to play Foo Fighters songs on ...
1,796528524030124618,positive,@AaqibAfzaal Pakistan may be an Islamic countr...
2,760964834217238632,positive,Happy Birthday to the coolest golfer in Bali! ...
3,147713180324524046,negative,@SimpplyA TMILLS is going to Tucson! But the 2...
4,732302280474120023,negative,Hmmmmm where are the #BlackLivesMatter when ma...
...,...,...,...
45020,660374218263817235,neutral,Sunday Cinema | Paul McCartney &amp; David Gil...
45021,739323365061217061,neutral,"14-Aug The day of independence, The day of S..."
45022,681369726697754114,positive,"9 September has arrived, which means Apple's n..."
45023,922217029064536808,positive,So I'll see you all tomorrow for some fun fill...


In [44]:
# test set
dftest = pd.read_csv("semeval-tweets/twitter-test1.txt", sep="\t", header=None, names=["id", "sentiment", "tweet"])
print(f"NaN tweets: {df['tweet'].isna().sum()}")
dftest = dftest.dropna(subset=["tweet"]).reset_index(drop=True)

for i in range(len(dftest["tweet"])):
  tweet = dftest.loc[i, "tweet"]
  tweetNoEmoji = dem.replace_with_desc(tweet)
  dftest.loc[i, "tweet"] = tweetNoEmoji
dftest

NaN tweets: 0


,id,sentiment,tweet
0,163361196206957578,neutral,Candids: Heading to the Chateau Marmont in Wes...
1,768006053969268950,negative,@Dont__KAY_me omg same I was reading it in sch...
2,742616104384772304,neutral,Watching MTV Hits! The Wanted Chasing the sun!
3,102313285628711403,neutral,"Bing one-ups knowledge graph, hires Encyclopae..."
4,653274888624828198,neutral,"On Thursday, concealed-carry gun license holde..."
...,...,...,...
3526,897427103459983282,positive,"@heavyoilcountry right you are, they could end..."
3527,494418545323766863,neutral,We will have to think about continuing in UPA:...
3528,556020888361859268,positive,ladies come throw that ass and niggas come thr...
3529,683639126147064420,neutral,@MandiieeKillaaa are you going to laidback Luk...


In [45]:
print(df["sentiment"].value_counts(dropna=False))

sentiment
neutral     20748
positive    15960
negative     8317
Name: count, dtype: int64


### Preprocess

In [46]:
# lower case everything
df["tweet"] = df['tweet'].str.lower()
dftest["tweet"] = dftest['tweet'].str.lower()
df

,id,sentiment,tweet
0,335104872099066692,positive,felt privileged to play foo fighters songs on ...
1,796528524030124618,positive,@aaqibafzaal pakistan may be an islamic countr...
2,760964834217238632,positive,happy birthday to the coolest golfer in bali! ...
3,147713180324524046,negative,@simpplya tmills is going to tucson! but the 2...
4,732302280474120023,negative,hmmmmm where are the #blacklivesmatter when ma...
...,...,...,...
45020,660374218263817235,neutral,sunday cinema | paul mccartney &amp; david gil...
45021,739323365061217061,neutral,"14-aug the day of independence, the day of s..."
45022,681369726697754114,positive,"9 september has arrived, which means apple's n..."
45023,922217029064536808,positive,so i'll see you all tomorrow for some fun fill...


In [47]:
# Remove urls
df["tweet"] = df['tweet'].str.replace(r'https?://\S+', '', regex=True)
dftest["tweet"] = dftest['tweet'].str.replace(r'https?://\S+', '', regex=True)

In [48]:
# Removing whitespace and \n
df["tweet"] = df["tweet"].str.replace(r'\s+', ' ', regex=True).str.strip()
dftest["tweet"] = dftest["tweet"].str.replace(r'\s+', ' ', regex=True).str.strip()

In [49]:
# remove punctuation
#df["tweet"] = df['tweet'].str.replace(r'[^\w\s]', '', regex=True)
#dftest["tweet"] = dftest['tweet'].str.replace(r'[^\w\s]', '', regex=True)
#df["tweet"].dtype

In [50]:
# Remove non english characters
for i in range(len(df["tweet"])):
  tweet = df.loc[i, "tweet"]
  df.loc[i, "tweet"] = tweet.encode('ascii',errors='ignore').decode('ascii')

for i in range(len(dftest["tweet"])):
  tweet = dftest.loc[i, "tweet"]
  dftest.loc[i, "tweet"] = tweet.encode('ascii',errors='ignore').decode('ascii')

In [51]:
# Lemmatisation with Texblob
for i in range(len(df["tweet"])):
    blob = tb.TextBlob(df.loc[i, "tweet"])
    lemmatised = []
    for word, tag in blob.tags:
        if tag.startswith('VB'):
            lemmatised.append(word.lemmatize('v'))
        elif tag.startswith('NN'):
            lemmatised.append(word.lemmatize('n'))
        elif tag.startswith('JJ'):
            lemmatised.append(word.lemmatize('a'))
        else:
            lemmatised.append(word)
    df.loc[i, "tweet"] = " ".join(lemmatised)

df

,id,sentiment,tweet
0,335104872099066692,positive,felt privilege to play foo fighter song on gui...
1,796528524030124618,positive,@ aaqibafzaal pakistan may be an islamic count...
2,760964834217238632,positive,happy birthday to the cool golfer in bali @ tj...
3,147713180324524046,negative,@ simpplya tmills be go to tucson but the 29th...
4,732302280474120023,negative,hmmmmm where be the blacklivesmatter when matt...
...,...,...,...
45020,660374218263817235,neutral,sunday cinema | paul mccartney & amp david gil...
45021,739323365061217061,neutral,14-aug the day of independence the day of sacr...
45022,681369726697754114,positive,9 september have arrive which mean apple 's ne...
45023,922217029064536808,positive,so i 'll see you all tomorrow for some fun fil...


In [52]:
for i in range(len(dftest["tweet"])):
    blob = tb.TextBlob(dftest.loc[i, "tweet"])
    lemmatised = []
    for word, tag in blob.tags:
        if tag.startswith('VB'):
            lemmatised.append(word.lemmatize('v'))
        elif tag.startswith('NN'):
            lemmatised.append(word.lemmatize('n'))
        elif tag.startswith('JJ'):
            lemmatised.append(word.lemmatize('a'))
        else:
            lemmatised.append(word)
    dftest.loc[i, "tweet"] = " ".join(lemmatised)

dftest

,id,sentiment,tweet
0,163361196206957578,neutral,candids heading to the chateau marmont in west...
1,768006053969268950,negative,@ dont__kay_me omg same i be read it in school...
2,742616104384772304,neutral,watch mtv hit the wanted chase the sun
3,102313285628711403,neutral,bing one-ups knowledge graph hire encyclopaedi...
4,653274888624828198,neutral,on thursday concealed-carry gun license holder...
...,...,...,...
3526,897427103459983282,positive,@ heavyoilcountry right you be they could end ...
3527,494418545323766863,neutral,we will have to think about continue in upa ma...
3528,556020888361859268,positive,lady come throw that as and niggas come throw ...
3529,683639126147064420,neutral,@ mandiieekillaaa be you go to laidback luke t...


### N-Gram and then BoW the dataset

In [53]:
df[y] = df[y].replace({"positive": 2, "neutral": 1, "negative": 0}).astype(int)
dftest[y] = dftest[y].replace({"positive": 2, "neutral": 1, "negative": 0}).astype(int)

# Classifier 1: SVM

In [54]:
X, y = "tweet", "sentiment"

In [66]:
Cs = [0.1,0.5,1.0,2.0]
bestC = 0
bestAcc = -np.inf

In [64]:
def modelFunc(c):
    pipe = skl.pipeline.Pipeline([
        ('tfidf', skl.feature_extraction.text.TfidfVectorizer(
            ngram_range=(1, 2), 
            sublinear_tf=True, 
            min_df=3, 
            max_df=0.95
        )),
        ('svm', skl.svm.LinearSVC(class_weight='balanced', C=c, max_iter=10000))
    ])
    
    pipe.fit(df[X], df[y])
    predictions = pipe.predict(dftest[X])  # or whatever your raw test text is
    acc = skl.metrics.accuracy_score(dftest[y], predictions)
    return acc, predictions

In [68]:
for c   bnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnn nnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnn nnnnnnnnnnnnnnnnnnnnnnm:
    acc, pred = modelFunc(c)         ~
    bestC = c if acc > bestAcc else bestC
    bestAcc = acc if acc > bestAcc else bestAcc

bestC, bestAcc

SyntaxError: invalid syntax (3710703325.py, line 1)

In [69]:
# Use the best model
skl.metrics.accuracy_score(dftest[y], predictions)

0.67487963749646

In [70]:
~

SyntaxError: invalid syntax (2415351536.py, line 1)

In [71]:
~

SyntaxError: invalid syntax (2415351536.py, line 1)

In [72]:
~~~~~~~

SyntaxError: invalid syntax (2822610422.py, line 1)

In [73]:
~

SyntaxError: invalid syntax (2415351536.py, line 1)

In [ ]:
 m